In [2]:
!pip -q install pandas altair vl-convert-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 68.3 MB/s eta 0:00:00


In [3]:
import pandas as pd

datos = [
    [2006, "Bajo", 391],
    [2006, "Medio bajo", 408],
    [2006, "Medio", 435],
    [2006, "Medio alto", 461],
    [2006, "Alto", 517],
    [2009, "Bajo", 406],
    [2009, "Medio bajo", 427],
    [2009, "Medio", 446],
    [2009, "Medio alto", 464],
    [2009, "Alto", 508],
    [2012, "Bajo", 395],
    [2012, "Medio bajo", 424],
    [2012, "Medio", 438],
    [2012, "Medio alto", 457],
    [2012, "Alto", 496],
    [2015, "Bajo", 414],
    [2015, "Medio bajo", 445],
    [2015, "Medio", 459],
    [2015, "Medio alto", 470],
    [2015, "Alto", 509],
    [2018, "Bajo", 409],
    [2018, "Medio bajo", 439],
    [2018, "Medio", 450],
    [2018, "Medio alto", 461],
    [2018, "Alto", 509],
    [2022, "Bajo", 413],
    [2022, "Medio bajo", 433],
    [2022, "Medio", 447],
    [2022, "Medio alto", 463],
    [2022, "Alto", 495],
]

df = pd.DataFrame(
    datos,
    columns=["anio", "quintil_socioeconomico", "puntaje_lectura"]
)

df.to_csv(
    "pisa_chile_lectura_quintiles_2006_2022.csv",
    index=False,
    encoding="utf-8-sig"
)

print(df.head())
print("Total de registros:", len(df))


   anio quintil_socioeconomico  puntaje_lectura
0  2006                   Bajo              391
1  2006             Medio bajo              408
2  2006                  Medio              435
3  2006             Medio alto              461
4  2006                   Alto              517
Total de registros: 30


In [4]:
df = pd.read_csv(
    "pisa_chile_lectura_quintiles_2006_2022.csv"
)

print(df.info())
print(df.isna().sum())
print(df.groupby("anio").size())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   anio                    30 non-null     int64 
 1   quintil_socioeconomico  30 non-null     object
 2   puntaje_lectura         30 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 852.0+ bytes
None
anio                      0
quintil_socioeconomico    0
puntaje_lectura           0
dtype: int64
anio
2006    5
2009    5
2012    5
2015    5
2018    5
2022    5
dtype: int64


In [5]:
extremos = df[
    df["quintil_socioeconomico"].isin(["Bajo", "Alto"])
]

tabla = extremos.pivot(
    index="anio",
    columns="quintil_socioeconomico",
    values="puntaje_lectura"
)

tabla["brecha"] = tabla["Alto"] - tabla["Bajo"]

print(tabla)


quintil_socioeconomico  Alto  Bajo  brecha
anio                                      
2006                     517   391     126
2009                     508   406     102
2012                     496   395     101
2015                     509   414      95
2018                     509   409     100
2022                     495   413      82


In [6]:
import altair as alt

orden = [
    "Bajo",
    "Medio bajo",
    "Medio",
    "Medio alto",
    "Alto"
]

colores = [
    "#237B75",
    "#72AAA3",
    "#9B9B9B",
    "#E7A46C",
    "#D75A4A"
]

base = alt.Chart(df).encode(
    x=alt.X(
        "anio:O",
        title="Año de evaluación",
        sort=[2006, 2009, 2012, 2015, 2018, 2022]
    ),
    y=alt.Y(
        "puntaje_lectura:Q",
        title="Puntaje promedio en lectura",
        scale=alt.Scale(domain=[350, 540])
    ),
    color=alt.Color(
        "quintil_socioeconomico:N",
        title="Quintil socioeconómico",
        scale=alt.Scale(
            domain=orden,
            range=colores
        ),
        sort=orden
    ),
    tooltip=[
        alt.Tooltip("anio:O", title="Año"),
        alt.Tooltip(
            "quintil_socioeconomico:N",
            title="Quintil"
        ),
        alt.Tooltip(
            "puntaje_lectura:Q",
            title="Puntaje",
            format=".0f"
        )
    ]
)

lineas = base.mark_line(
    strokeWidth=3
)

puntos = base.mark_circle(
    size=85
)

grafico = (
    (lineas + puntos)
    .properties(
        width=750,
        height=420,
        title=alt.TitleParams(
            text="La distancia lectora entre quintiles se estrecha",
            subtitle=[
                "Chile, PISA 2006–2022",
                "La brecha entre el quintil más alto y el más bajo pasó de 126 a 82 puntos."
            ],
            anchor="start"
        )
    )
    .interactive()
)

grafico

alt.LayerChart(...)

In [8]:
grafico.save(
    "visualizacion_pisa_socioeconomico.html"
)

grafico.save(
    "visualizacion_pisa_socioeconomico.png",
    scale_factor=2
)

from PIL import Image

imagen = Image.open(
    "visualizacion_pisa_socioeconomico.png"
).convert("RGB")

imagen.save(
    "visualizacion_pisa_socioeconomico.jpg",
    quality=95
)

print("Archivos exportados correctamente")


Archivos exportados correctamente
